In [ ]:
import networkx as nx
import heapq
import time
from scipy.io import mmread
import matplotlib.pyplot as plt




==========================
Load Graph
==========================


In [ ]:

def load_graph(file_path):

    if file_path.endswith(".mtx"):

        matrix = mmread(file_path)

        G = nx.from_scipy_sparse_array(matrix)

    elif file_path.endswith(".edges"):

        G = nx.Graph()

        with open(file_path, "r") as file:

            for line in file:

                if line.startswith("%"):
                    continue

                parts = line.split()

                if len(parts) >= 2:

                    u = int(parts[0])
                    v = int(parts[1])

                    G.add_edge(u, v)

    else:
        raise ValueError(f"Unsupported file format: {file_path}")

    return G




==========================
Bidirectional Dijkstra
==========================


In [ ]:

def bidirectional_dijkstra(G, source, target):

    if source == target:
        return 0, [source]

    dist_fwd = {node: float('inf') for node in G.nodes()}
    dist_bwd = {node: float('inf') for node in G.nodes()}

    dist_fwd[source] = 0
    dist_bwd[target] = 0

    pq_fwd = [(0, source)]
    pq_bwd = [(0, target)]

    visited_fwd = {}
    visited_bwd = {}

    best = float('inf')

    visited_order = []

    while pq_fwd and pq_bwd:

        # forward step
        if pq_fwd:
            d1, u = heapq.heappop(pq_fwd)

            if u not in visited_fwd:
                visited_fwd[u] = d1
                visited_order.append(u)

                if u in visited_bwd:
                    best = min(best, d1 + visited_bwd[u])

                for v in G.neighbors(u):

                    w = G[u][v].get("weight", 1)

                    if dist_fwd[v] > d1 + w:

                        dist_fwd[v] = d1 + w
                        heapq.heappush(pq_fwd, (dist_fwd[v], v))

        # backward step
        if pq_bwd:
            d2, u = heapq.heappop(pq_bwd)

            if u not in visited_bwd:
                visited_bwd[u] = d2

                if u in visited_fwd:
                    best = min(best, d2 + visited_fwd[u])

                for v in G.neighbors(u):

                    w = G[u][v].get("weight", 1)

                    if dist_bwd[v] > d2 + w:

                        dist_bwd[v] = d2 + w
                        heapq.heappush(pq_bwd, (dist_bwd[v], v))

    return best, visited_order




==========================
DATASETS
==========================


In [ ]:

datasets = [
    "inf-USAir97.mtx",
    "inf-power.mtx",
    "road-roadNet-CA.mtx",
    "road-euroroad.edges"
]




==========================
MAIN
==========================


In [ ]:

if __name__ == "__main__":

    print("\n===== BIDIRECTIONAL DIJKSTRA RESULTS =====\n")

    dataset_names = []
    execution_times = []
    node_counts = []
    edge_counts = []

    for dataset in datasets:

        print("=" * 60)
        print("Dataset:", dataset)

        G = load_graph(dataset)

        print("Nodes:", G.number_of_nodes())
        print("Edges:", G.number_of_edges())

        source = list(G.nodes())[0]
        target = list(G.nodes())[-1]

        start = time.perf_counter()

        distance, order = bidirectional_dijkstra(G, source, target)

        end = time.perf_counter()

        execution_time = end - start

        print("Execution Time:", round(execution_time, 6), "seconds")
        print("Visited Nodes:", len(order))
        print()

        dataset_names.append(dataset)
        execution_times.append(execution_time)
        node_counts.append(G.number_of_nodes())
        edge_counts.append(G.number_of_edges())




==========================
Graphs
==========================


In [ ]:

    plt.figure(figsize=(8, 5))
    plt.plot(dataset_names, execution_times, marker="o")
    plt.title("Bidirectional Dijkstra Execution Time")
    plt.xlabel("Datasets")
    plt.ylabel("Time (seconds)")
    plt.grid()
    plt.savefig("bidirectional_dijkstra_time.png")
    plt.show()


    plt.figure(figsize=(8, 5))
    plt.bar(dataset_names, node_counts)
    plt.title("Node Growth (Bidirectional Dijkstra)")
    plt.xlabel("Datasets")
    plt.ylabel("Nodes")
    plt.grid()
    plt.savefig("bidirectional_node_growth.png")
    plt.show()


    plt.figure(figsize=(8, 5))
    plt.bar(dataset_names, edge_counts)
    plt.title("Edge Growth (Bidirectional Dijkstra)")
    plt.xlabel("Datasets")
    plt.ylabel("Edges")
    plt.grid()
    plt.savefig("bidirectional_edge_growth.png")
    plt.show()
